In [1]:
from pyspark.sql import functions as F

# Read cleaned ECB data from Silver Lakehouse
fx_df = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Silver_LH.dbo.ecb_usd_eur_clean"
)

print("Silver FX records:", fx_df.count())

# Build Gold FX dimension
dim_fx = (
    fx_df
    .select(
        "date",
        "usd_per_eur",
        "year",
        "month"
    )
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "eur_per_usd",
        F.round(1 / F.col("usd_per_eur"), 6)
    )
    .dropDuplicates(["date"])
    .orderBy("date")
)

# Save to Gold Lakehouse
(
    dim_fx.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("DimFX")
)

print("Gold FX dimension saved successfully!")
print("Gold records:", dim_fx.count())

display(dim_fx.orderBy(F.desc("date")).limit(10))

StatementMeta(, 861c6dc7-a8e1-4e26-83f9-50fca9903682, 3, Finished, Available, Finished, False)

Silver FX records: 7127
Gold FX dimension saved successfully!
Gold records: 7127


SynapseWidget(Synapse.DataFrame, 28cfe313-22b5-464e-ba7a-d80923cb9b7d)